# convT-kernel-axis-swap — worked example 2: Reproduce ConvTranspose1d with a flipped, axis-swapped Conv1d kernel

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-kernel-axis-swap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A stride-1 `ConvTranspose1d` is mathematically a *full* (zero-padded) `Conv1d` whose kernel has been (a) channel-axis-swapped `(IC, OC, K) -> (OC, IC, K)` and (b) spatially flipped. The axis swap converts the transposed-conv `(in, out, k)` storage into conv `(out, in, k)` storage; the flip converts correlation into the true convolution the transpose implements.

## Worked solution

**Goal.** Given a `ConvTranspose1d` kernel in `(IC, OC, K)` layout and an input, reproduce the module's output using only `F.conv1d`.

**Step 1 — swap the channel axes.** `ConvTranspose1d` stores its weight as `(IC, OC, K)`. `F.conv1d` expects `(OC, IC, K)`, so apply `rearrange(w, 'i o k -> o i k')` to move out-channels to the front. Spatial axis `k` stays put.

**Step 2 — flip the spatial axis.** Transposed convolution applies the kernel in the *reversed* spatial order relative to cross-correlation (which is what PyTorch's conv ops actually compute). So reverse the last axis with `t.flip(..., dims=[2])`. Together, steps 1 and 2 are the `'i o k -> o i k'` swap *after* the flip described in the atom definition.

**Step 3 — pad to a *full* convolution.** A transposed conv with kernel size `K` and stride 1 grows the signal: output length = input length + K − 1. A plain `conv1d` shrinks it. To match, pad by `K - 1` on each side (`padding=K-1`), which gives the full convolution and the same output length.

**Step 4 — verify numerically.** Build the real `nn.ConvTranspose1d`, copy the weight in, and compare. `t.allclose` confirms the flipped + swapped + full-padded conv reproduces the transposed conv to floating-point tolerance.

In [ ]:
import torch.nn.functional as F
def convT1d_via_full_conv(x: Tensor, w_convT: Tensor) -> Tensor:
    k = w_convT.shape[-1]
    # axis swap (in,out -> out,in) AND spatial flip, then full (K-1) padding
    w_conv = t.flip(rearrange(w_convT, 'i o k -> o i k'), dims=[2])
    return F.conv1d(x, w_conv, padding=k - 1)

t.manual_seed(0)
ic, oc, k = 2, 3, 5
w_convT = t.randn(ic, oc, k)          # (IC, OC, K)
x = t.randn(1, ic, 7)
ct = t.nn.ConvTranspose1d(ic, oc, k, bias=False)
ct.weight.data.copy_(w_convT)
y_ref = ct(x)
y_got = convT1d_via_full_conv(x, w_convT)
print('ConvTranspose1d output shape:', tuple(y_ref.shape))
print('full-conv reconstruction matches:', t.allclose(y_got, y_ref, atol=1e-5))